# Lab 4. Anonymity

## Implementation

In [2]:
import pandas as pd
import numpy as np
from typing import Union, List, Dict, Tuple

### Data handling functions

In [3]:
def build_toy_dataset(**kwargs):
    '''
    Builds a toy dataset with fixed records.
    
    Returns:
    - A tuple containing:
        * Dataset as a Pandas Dataframe
        * List of quasi-identifiers
        * Sensitive column name
    '''
    data = [
        [6, "1", "test1", "x", 20],
        [6, "1", "test1", "x", 30],
        [8, "2", "test2", "x", 50],
        [8, "2", "test3", "w", 45],
        [8, "1", "test2", "y", 35],
        [4, "2", "test3", "y", 20],
        [4, "1", "test3", "y", 20],
        [2, "1", "test3", "z", 22],
        [2, "2", "test3", "y", 32],
    ]

    columns = ["col1", "col2", "col3", "col4", "col5"]
    categorical = set(("col2", "col3", "col4"))

    df = pd.DataFrame(data=data, columns=columns)


    for name in categorical:
        df[name] = df[name].astype("category")

    return df, ["col1", "col2", "col3"], 'col4'

In [4]:
def generate_random_dataset(n: int=200, **kwargs):
    '''
    Generates a toy dataset containing n distinct samples.

    - n: number of samples to generate

    Returns:
    - A tuple containing:
        * Dataset as a Pandas Dataframe
        * List of quasi-identifiers
        * Sensitive column name
    '''
    diseases = np.array(["Angine", "Appendicite", "Chlamydia", "Cataracte", "Dengue", 
                         "Eczéma", "Grippe", "Hépatite B", "Hépatite C", "Rhino-pharyngite", 
                         "Otite", "Rougeole", "Scarlatine", "Urticaire", "Varicelle", "Zona"])
    zipcodes = np.array([35000, 35200, 37000, 40000, 40500, 50000, 52000, 60000, 62000, 68000, 
                         75000, 75001, 75002, 75005])

    rows = []
    for _ in range(n):
        row = {'Age':np.random.randint(7, 77), 'ZipCode':np.random.choice(zipcodes), 'Disease':np.random.choice(diseases)}
        while row in rows:
            row = {'Age':np.random.randint(7, 77), 'ZipCode':np.random.choice(zipcodes), 'Disease':np.random.choice(diseases)}
        rows.append(row)
        
        
    dataset = pd.DataFrame(rows)
    dataset.sort_values(by = ['Age', 'ZipCode'], inplace=True)

    return dataset, ['Age', 'ZipCode'], 'Disease'

In [5]:
def read_adult(path: str, **kwargs) -> pd.DataFrame:
    '''
    Reads the adult dataset.

    Parameters:
        - path: path to the CSV file
    
    Returns:
    - A tuple containing:
        * Dataset as a Pandas Dataframe
        * List of quasi-identifiers
        * Sensitive column name
    '''

    df = pd.read_csv(path, header=0, index_col=None, sep=',')
    categorical = ['workclass', 'education', 'marital.status', 'occupation',
        'race', 'sex', 'native.country']
    for name in categorical:
        df[name] = df[name].astype("category")

    return df, ['age', 'workclass', 'education', 'marital.status', 'occupation',
        'race', 'sex', 'native.country'], 'income'

In [6]:
def build_data(type='toy', **kwargs):
    '''
    Build data

    Parameters:
        - type: toy, random or adult
        - kwargs: arguments for the underlying data generation functions
    
    Returns:
    - A tuple containing:
        * Dataset as a Pandas Dataframe
        * List of quasi-identifiers
        * Sensitive column name
    '''

    if type == 'toy':
        return build_toy_dataset()
    elif type == 'random':
        return generate_random_dataset(**kwargs)
    elif type == 'adult':
        return read_adult(**kwargs)
    
    return None

### Anonymity functions

In [7]:
def get_k(df: pd.DataFrame, quasi_identifiers: list[str]):
    '''
    Obtains the number of different rows within a partition.

    Parameters:
        - df: a dataframe with the columns to analyze
        - quasi_identifiers: a list with the quasi identifier columns

    Returns:
        - K parameter for k-anonymity
    '''
    if all(quasi_id in df.columns for quasi_id in quasi_identifiers):
        return len(df[quasi_identifiers].drop_duplicates())
    else:
        raise ValueError(f"{quasi_identifiers} columns not contained in the provided Dataframe")

def get_l(df: pd.DataFrame, sensitive_column: str):
    '''
    Gets the number of different sensitive values within a partition.

    Parameters:
        - df: a dataframe with the columns to analyze
        - sensitive_column: the column with the sensitive values

    Returns:
        - L parameter for l-diversity
    '''
    if sensitive_column in df.columns:
        return len(df[sensitive_column].drop_duplicates())
    else:
        raise ValueError(f"{sensitive_column} column not contained in the provided Dataframe")

def get_t(df: pd.DataFrame, sensitive_column: str, frequences: pd.Series):
    '''
    Gets the distance between the distribution of the sensitive column within
    the partition with respect to the global distribution.

    Parameters:
        - df: a dataframe with the columns to analyze
        - sensitive_column: the column with the sensitive values
        - frequences: a series with the frequences of the whole dataframe
    
    Returns:
        - T parameter for t-closeness
    '''
    if sensitive_column not in df.columns:
        raise ValueError(f"{sensitive_column} column not contained in the provided Dataframe")

    part_freq = df[sensitive_column].value_counts().sort_index() / df.shape[0]

    if df.dtypes[sensitive_column] == 'category':
        emd = 0
        for col in frequences.index:
            emd += abs(frequences[col] - part_freq[col])
        
        return 1/2 * emd

    else:
        emd = 0
        aux_emd = 0
        for col in frequences.index:
            if col not in part_freq.index:
                aux_emd = abs(aux_emd + (0 - frequences[col]))
                emd += emd + aux_emd
            else:
                aux_emd = abs(aux_emd + (part_freq[col] - frequences[col]))
                emd += emd + aux_emd

        return (1/(frequences.count()-1)) * emd

In [8]:
def is_k_anonymous(df: pd.DataFrame, quasi_identifiers: list[str], k: int):
    '''
    Checks if a partition is k-anonymous.

    Parameters:
        - df: a dataframe with the columns to analyze
        - quasi_identifiers: a list with the quasi identifier columns
        - k: number of different rows per partition

    Returns:
        - True if the partition satisfies k-anonymity. False otherwise
    '''
    return get_k(df, quasi_identifiers) >= k

def is_l_diverse(df: pd.DataFrame, sensitive_column: str, l: int):
    '''
    Checks if a partition is l-diverse.

    Parameters:
        - df: a dataframe with the columns to analyze
        - sensitive_column: the column with the sensitive values
        - l: number of distinct values for the sensitive column within the partition

    Returns:
        - True if the partition satisfies l-diversity. False otherwise
    '''
    return get_l(df, sensitive_column) >= l

def is_t_close(df: pd.DataFrame, sensitive_column: str, frequences: pd.Series, t: float):
    '''
    Checks if a partition is t-close.

    Parameters:
        - df: a dataframe with the columns to analyze
        - sensitive_column: the column with the sensitive values
        - frequences: a series with the frequences of the whole dataframe
        - t: distance between the distribution of the sensitive column within the partition and the global (true) distribution

    Returns:
        - True if the partition satisfies t-closeness. False otherwise
    '''
    return get_t(df, sensitive_column, frequences) <= t

def is_valid(partition: pd.DataFrame, quasi_identifiers: list[str], sensitive_column: str, frequences: pd.Series, k: int, l: Union[None, int] = None, t: Union[None, float] = None):
    '''
    Checks if a partition is valid according to the required anonymity.

    Parameters:
        - df: a dataframe with the columns to analyze
        - quasi_identifiers: a list with the quasi identifier columns
        - sensitive_column: the column with the sensitive values
        - frequences: a series with the frequences of the whole dataframe
        - k: number of different rows per partition
        - l: number of distinct values for the sensitive column within the partition
        - t: distance between the distribution of the sensitive column within the partition and the global (true) distribution

    Returns:
        - True if the partition satisfies t-closeness. False otherwise
    '''
    if l == None and t == None:
        return is_k_anonymous(partition, quasi_identifiers, k)
    elif t == None:
        return is_k_anonymous(partition, quasi_identifiers, k) and is_l_diverse(partition, sensitive_column, l)
    elif l == None:
        return is_k_anonymous(partition, quasi_identifiers, k) and is_t_close(partition, sensitive_column, frequences, t)
    else:
        return is_k_anonymous(partition, quasi_identifiers, k) and is_l_diverse(partition, sensitive_column, l) and is_t_close(partition, sensitive_column, frequences, t)

In [9]:
def analyze_dimensions(dataframe: pd.DataFrame, quasi_identifiers: list[str], main_widths: Union[Dict[str, float], None] = None) -> Dict[str, float]:
    '''
    Analyzes the width of each quasi-identifier column within the partition to decide
    on which one to perform the partitioning.
    '''
    widths = {}
    for quasi_id in quasi_identifiers:
        if dataframe[quasi_id].dtype.kind in 'iufc':  # Numeric types
            partition_width = dataframe[quasi_id].max() - dataframe[quasi_id].min()
            global_width = main_widths[quasi_id] if main_widths else partition_width
            widths[quasi_id] = partition_width / global_width if global_width != 0 else 0
        else:  # Categorical types
            partition_width = len(dataframe[quasi_id].unique())
            global_width = main_widths[quasi_id] if main_widths else partition_width
            widths[quasi_id] = partition_width / global_width if global_width != 0 else 0

    return dict(sorted(widths.items(), key=lambda x: -x[1]))

In [10]:
def split(partition: pd.DataFrame, column: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    '''
    Splits the partition on the specified quasi-identifier column.
    '''
    col = partition[column]

    if col.dtype.kind in 'iufc':  # Numeric types
        median_value = col.median()
        left_partition = partition[col <= median_value]
        right_partition = partition[col > median_value]
    else:  # Categorical types
        unique_values = sorted(col.unique())
        mid_index = len(unique_values) // 2
        left_partition = partition[col.isin(unique_values[:mid_index])]
        right_partition = partition[col.isin(unique_values[mid_index:])]

    return left_partition, right_partition

In [11]:
def build_partitions(database: pd.DataFrame, quasi_identifiers: List[str], sensitive_column: str, 
                     frequencies: pd.Series, k: int, l: Union[int, None] = None, t: Union[float, None] = None) -> List[pd.DataFrame]:
    '''
    Builds partitions to satisfy k-anonymity, l-diversity, and t-closeness.
    '''
    partitions = []
    queue = [database]

    # Calculate global widths once
    main_widths = analyze_dimensions(database, quasi_identifiers)

    while queue:
        partition = queue.pop(0)
        widths = analyze_dimensions(partition, quasi_identifiers, main_widths)

        for qi in widths:
            left_partition, right_partition = split(partition, qi)

            if is_valid(left_partition, quasi_identifiers, sensitive_column, frequencies, k, l, t) and \
               is_valid(right_partition, quasi_identifiers, sensitive_column, frequencies, k, l, t):
                queue.extend([left_partition, right_partition])
                break
        else:
            partitions.append(partition)
    return partitions

In [12]:
def generalize_numerical(column) -> str:
    '''
    Generalizes a numerical column by computing the min-max range.

    Parameters:
        - column: name of the column to be generalized

    Returns:
        - Generalized value for the entire sample
    '''

    min_val = column.min()
    max_val = column.max()
    return f"{min_val}-{max_val}" if min_val != max_val else f"{min_val}"
    
def generalize_categorical(column) -> str:
    '''
    Generalizes a categorical column by grouping all the possible values.

    Parameters:
        - column: name of the column to be generalized

    Returns:
        - Generalized value for the entire sample
    '''

    return ','.join(column.unique().astype(str))


def generalize(partitions: List[pd.DataFrame], quasi_identifiers: List[str], sensitive_column: str) -> pd.DataFrame:
    '''
    Generalizes each one of the partitions defined for the database.
    '''
    result = []

    for num_partition, partition in enumerate(partitions):
        generalized_row = {}
        for qi in quasi_identifiers:
            col = partition[qi]
            if col.nunique() > 1:
                if col.dtype.kind in 'iufc':  # Numeric types
                    generalized_row[qi] = f"{col.min()}-{col.max()}"
                else:  # Categorical types
                    generalized_row[qi] = ','.join(sorted(map(str, col.unique())))
            else:
                generalized_row[qi] = col.iloc[0]  # No generalization needed

        for ix in partition.index:
            row = {qi: generalized_row[qi] for qi in quasi_identifiers}
            row[sensitive_column] = partition.loc[ix, sensitive_column]
            row['index'] = ix
            row['partition'] = num_partition
            result.append(row)

    df = pd.DataFrame(result)
    df.set_index('index', inplace=True, drop=True)
    df.index.name = None
    return df

In [13]:
def anonymize(database: pd.DataFrame, quasi_identifiers: List[str], sensitive_column: str, 
              k: int, l: Union[int, None] = None, t: Union[float, None] = None) -> pd.DataFrame:
    '''
    Anonymizes a database according to the required parameters.

    Parameters:
        - database: entire database from which the partition is extracted
        - quasi_identifiers: names of the quasi-identifier columns
        - sensitive_column: name of the sensitive column
        - k: number of different rows per partition
        - l: number of distinct values for the sensitive column within the partition
        - t: Earth Mover's Distance between the partition and the global distribution

    Returns:
        - DataFrame with generalized values for each partition
    '''
    frequencies = database[sensitive_column].value_counts(normalize=True).sort_index()

    # Calculate global widths only once
    global_widths = analyze_dimensions(database, quasi_identifiers)

    partitions = build_partitions(database, quasi_identifiers, sensitive_column, frequencies, k, l, t)
    return generalize(partitions, quasi_identifiers, sensitive_column)

## Example

In [14]:
df, quasi_identifiers, sensitive_column = build_data(type='adult', n=5000, path='E:/MUNICS/PAN/anonymity_data/anonymity_data/adult.csv')

In [15]:
df

,age,workclass,education,marital.status,occupation,race,sex,native.country,income
0,90,?,HS-grad,Widowed,?,White,Female,United-States,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,White,Female,United-States,<=50K
2,66,?,Some-college,Widowed,?,Black,Female,United-States,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,White,Female,United-States,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,White,Female,United-States,<=50K
...,...,...,...,...,...,...,...,...,...
32556,22,Private,Some-college,Never-married,Protective-serv,White,Male,United-States,<=50K
32557,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,White,Female,United-States,<=50K
32558,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,White,Male,United-States,>50K
32559,58,Private,HS-grad,Widowed,Adm-clerical,White,Female,United-States,<=50K


In [16]:
anonymize(df, quasi_identifiers, sensitive_column, k=2)

,age,workclass,education,marital.status,occupation,race,sex,native.country,income,partition
15249,34-35,Private,"10th,12th",Married-civ-spouse,"Craft-repair,Other-service","Asian-Pac-Islander,White",Male,"Guatemala,India",<=50K,0
27141,34-35,Private,"10th,12th",Married-civ-spouse,"Craft-repair,Other-service","Asian-Pac-Islander,White",Male,"Guatemala,India",<=50K,0
9639,45-46,"Private,Self-emp-not-inc","7th-8th,9th,Assoc-acdm",Married-civ-spouse,"Adm-clerical,Exec-managerial,Handlers-cleaners","Asian-Pac-Islander,White",Male,"Greece,India",<=50K,1
10253,45-46,"Private,Self-emp-not-inc","7th-8th,9th,Assoc-acdm",Married-civ-spouse,"Adm-clerical,Exec-managerial,Handlers-cleaners","Asian-Pac-Islander,White",Male,"Greece,India",<=50K,1
21602,45-46,"Private,Self-emp-not-inc","7th-8th,9th,Assoc-acdm",Married-civ-spouse,"Adm-clerical,Exec-managerial,Handlers-cleaners","Asian-Pac-Islander,White",Male,"Greece,India",<=50K,1
...,...,...,...,...,...,...,...,...,...,...
11879,29-30,Private,HS-grad,Never-married,Other-service,White,Male,United-States,<=50K,8029
16780,29-30,Private,HS-grad,Never-married,Other-service,White,Male,United-States,<=50K,8029
27146,29-30,Private,HS-grad,Never-married,Other-service,White,Male,United-States,<=50K,8029
28455,29-30,Private,HS-grad,Never-married,Other-service,White,Male,United-States,<=50K,8029


In [17]:
anonymize(df, quasi_identifiers, sensitive_column, k=2, l=2)

,age,workclass,education,marital.status,occupation,race,sex,native.country,income,partition
1004,17-29,"?,Private,Self-emp-inc","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th","Divorced,Married-civ-spouse,Married-spouse-abs...","?,Adm-clerical,Craft-repair,Exec-managerial,Fa...","Asian-Pac-Islander,Black,Other,White","Female,Male","?,Canada,China,Columbia,Dominican-Republic,Ecu...",<=50K,0
1117,17-29,"?,Private,Self-emp-inc","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th","Divorced,Married-civ-spouse,Married-spouse-abs...","?,Adm-clerical,Craft-repair,Exec-managerial,Fa...","Asian-Pac-Islander,Black,Other,White","Female,Male","?,Canada,China,Columbia,Dominican-Republic,Ecu...",<=50K,0
4004,17-29,"?,Private,Self-emp-inc","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th","Divorced,Married-civ-spouse,Married-spouse-abs...","?,Adm-clerical,Craft-repair,Exec-managerial,Fa...","Asian-Pac-Islander,Black,Other,White","Female,Male","?,Canada,China,Columbia,Dominican-Republic,Ecu...",<=50K,0
4661,17-29,"?,Private,Self-emp-inc","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th","Divorced,Married-civ-spouse,Married-spouse-abs...","?,Adm-clerical,Craft-repair,Exec-managerial,Fa...","Asian-Pac-Islander,Black,Other,White","Female,Male","?,Canada,China,Columbia,Dominican-Republic,Ecu...",<=50K,0
5893,17-29,"?,Private,Self-emp-inc","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th","Divorced,Married-civ-spouse,Married-spouse-abs...","?,Adm-clerical,Craft-repair,Exec-managerial,Fa...","Asian-Pac-Islander,Black,Other,White","Female,Male","?,Canada,China,Columbia,Dominican-Republic,Ecu...",<=50K,0
...,...,...,...,...,...,...,...,...,...,...
20462,41-42,Private,Bachelors,Married-civ-spouse,Exec-managerial,White,Male,"United-States,Yugoslavia",<=50K,3527
23394,41-42,Private,Bachelors,Married-civ-spouse,Exec-managerial,White,Male,"United-States,Yugoslavia",>50K,3527
24801,41-42,Private,Bachelors,Married-civ-spouse,Exec-managerial,White,Male,"United-States,Yugoslavia",>50K,3527
25609,41-42,Private,Bachelors,Married-civ-spouse,Exec-managerial,White,Male,"United-States,Yugoslavia",>50K,3527


In [19]:
anonymize(df, quasi_identifiers, sensitive_column, k=2, l=2, t=0.2)

,age,workclass,education,marital.status,occupation,race,sex,native.country,income,partition
27,17-81,"Private,Self-emp-inc,Self-emp-not-inc,State-gov","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Ass...","Divorced,Married-civ-spouse,Married-spouse-abs...","Adm-clerical,Craft-repair,Exec-managerial,Farm...","Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Ot...","Female,Male","France,Germany,Greece,Guatemala,Haiti,Holand-N...",<=50K,0
67,17-81,"Private,Self-emp-inc,Self-emp-not-inc,State-gov","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Ass...","Divorced,Married-civ-spouse,Married-spouse-abs...","Adm-clerical,Craft-repair,Exec-managerial,Farm...","Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Ot...","Female,Male","France,Germany,Greece,Guatemala,Haiti,Holand-N...",>50K,0
90,17-81,"Private,Self-emp-inc,Self-emp-not-inc,State-gov","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Ass...","Divorced,Married-civ-spouse,Married-spouse-abs...","Adm-clerical,Craft-repair,Exec-managerial,Farm...","Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Ot...","Female,Male","France,Germany,Greece,Guatemala,Haiti,Holand-N...",>50K,0
211,17-81,"Private,Self-emp-inc,Self-emp-not-inc,State-gov","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Ass...","Divorced,Married-civ-spouse,Married-spouse-abs...","Adm-clerical,Craft-repair,Exec-managerial,Farm...","Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Ot...","Female,Male","France,Germany,Greece,Guatemala,Haiti,Holand-N...",<=50K,0
214,17-81,"Private,Self-emp-inc,Self-emp-not-inc,State-gov","10th,11th,12th,1st-4th,5th-6th,7th-8th,9th,Ass...","Divorced,Married-civ-spouse,Married-spouse-abs...","Adm-clerical,Craft-repair,Exec-managerial,Farm...","Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Ot...","Female,Male","France,Germany,Greece,Guatemala,Haiti,Holand-N...",>50K,0
...,...,...,...,...,...,...,...,...,...,...
28961,51-57,Self-emp-not-inc,HS-grad,Married-civ-spouse,Craft-repair,"Black,White",Male,United-States,>50K,138
29326,51-57,Self-emp-not-inc,HS-grad,Married-civ-spouse,Craft-repair,"Black,White",Male,United-States,<=50K,138
31337,51-57,Self-emp-not-inc,HS-grad,Married-civ-spouse,Craft-repair,"Black,White",Male,United-States,<=50K,138
31576,51-57,Self-emp-not-inc,HS-grad,Married-civ-spouse,Craft-repair,"Black,White",Male,United-States,<=50K,138


# Discussion

Briefly discuss how the different values of *k*, *l*, and *t* affect the given dataset.

Sabemos que K define el nivel de anonimizacion mediante las particiones

Al definir k = 2  

Vemos que los valores de la columna 1 ahora se agrupan en rangos 2-4, 2-6 y 8 de manera que se garantice que al menos haya 2 registros en cada particion.

Luego en la columna 3 se combinan los valores que se encuentran dentro de un rango de la columna 1 se combinan en listas, indicando que los registros originales tenian multiples valores posibles en esa categoria.

En este caso el dataset se divide en 3 particiones como se puede apreciar en la tabla resultante donde esta la particion col1=8 col1=2-4 y col1=2-6.

Ahora en el Caso donde incluimos L-diversidad:

k=2 y l=2

Ahora las particiones deben contener valores diversos en col2 y col3 

En la particion 2-6 la col2 incluye valores 1 y 2 y en col3 incluye test1 y test3 haciendo asi que sea menos identificable cada una de las particiones

En este caso se reducen la cantidad de particiones, resultando en este caso solo 2 esto debido a que la diversificacion obliga que se agrupen mas registros juntos para poder cumplir con las condiciones de l.

Esto produce que dentro de cada particion los valores de las columnas resulten mas variados dificultando asi que dentro de un mismo grupo se pueda saber exactamente que registro original era.

Ahora, para el caso donde introducimos t-closeness 

k=2 l=2 y t=0.2

El cambio significativo se observa en la col1 donde el rango ahora abarca todas las opciones posibles 2-8, generando que todos los registros se agrupen en una sola particion independientemente de las diferencias entre sus valores originales.

Ademas, ahora se aprecia que solo quedan dos particiones aquellas cuyo valor de col2 es 1 y aquellas donde el valor es 2.

En este caso se aumenta la privacidad pero tambien se reduce la usabilidad del dataset de manera importante. Los datos quedan sumamente generalizados y y pudiera afectar cuando necesita realizarse un analisis mas preciso.


Podemos afirmar que a medida que anadimos un parametro la privacidad aumenta y las posibilidades de inferencia se reduce. A medida que se agregan restricciones el dataset perdera especifidad lo que podria afectar analisis donde se necesita precision.

En general se debe buscar un balance entre k,l y t para que se garantice la privacidad manteniendo un nivel de utilizacion de los datos que permita realizar los analisis necesarios


